In [1]:
import psutil
import torch
import numpy as np
import multiprocessing as mp

print("="*80)
print("EMERGENCY DIAGNOSTICS")
print("="*80)

# 1. CPU Information
print(f"\n1. CPU INFO:")
print(f"   Physical cores: {psutil.cpu_count(logical=False)}")
print(f"   Logical cores: {psutil.cpu_count(logical=True)}")
print(f"   Current CPU usage: {psutil.cpu_percent(interval=2)}%")
print(f"\n   Per-core usage:")
for i, usage in enumerate(psutil.cpu_percent(interval=1, percpu=True)):
    print(f"      Core {i}: {usage:5.1f}%")

# 2. Memory
mem = psutil.virtual_memory()
print(f"\n2. MEMORY:")
print(f"   Total: {mem.total / (1024**3):.1f} GB")
print(f"   Used: {mem.used / (1024**3):.1f} GB ({mem.percent}%)")
print(f"   Available: {mem.available / (1024**3):.1f} GB")

# 3. Python Processes
print(f"\n3. PYTHON PROCESSES:")
python_count = 0
for proc in psutil.process_iter(['pid', 'name', 'cpu_percent', 'memory_percent', 'num_threads']):
    try:
        if 'python' in proc.info['name'].lower():
            python_count += 1
            print(f"   PID {proc.info['pid']}: CPU={proc.info['cpu_percent']:5.1f}% "
                  f"RAM={proc.info['memory_percent']:5.1f}% Threads={proc.info['num_threads']}")
    except:
        pass
print(f"   Total Python processes: {python_count}")

# 4. PyTorch Configuration
print(f"\n4. PYTORCH CONFIG:")
print(f"   PyTorch version: {torch.__version__}")
print(f"   Num threads: {torch.get_num_threads()}")
print(f"   MKL available: {torch.backends.mkl.is_available()}")

# 5. NumPy Configuration  
print(f"\n5. NUMPY CONFIG:")
print(f"   NumPy version: {np.__version__}")
try:
    np.show_config()
except:
    print("   Could not show NumPy config")

# 6. Environment Variables
print(f"\n6. THREADING ENVIRONMENT VARIABLES:")
import os
thread_vars = ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS', 
               'OPENBLAS_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS']
for var in thread_vars:
    value = os.environ.get(var, 'NOT SET')
    print(f"   {var}: {value}")

print("="*80)

EMERGENCY DIAGNOSTICS

1. CPU INFO:
   Physical cores: 4
   Logical cores: 8
   Current CPU usage: 28.3%

   Per-core usage:
      Core 0:  50.0%
      Core 1:  22.7%
      Core 2:  38.5%
      Core 3:  18.8%
      Core 4:  35.8%
      Core 5:  17.2%
      Core 6:  30.8%
      Core 7:  27.3%

2. MEMORY:
   Total: 15.8 GB
   Used: 8.6 GB (54.6%)
   Available: 7.2 GB

3. PYTHON PROCESSES:
   PID 9224: CPU=  0.0% RAM=  0.1% Threads=1
   PID 12708: CPU=  0.0% RAM=  0.0% Threads=1
   PID 17960: CPU=  0.0% RAM=  1.4% Threads=24
   Total Python processes: 3

4. PYTORCH CONFIG:
   PyTorch version: 2.8.0+cpu
   Num threads: 4
   MKL available: True

5. NUMPY CONFIG:
   NumPy version: 2.3.3
Build Dependencies:
  blas:
    detection method: pkgconfig
    found: true
    include directory: C:/Users/runneradmin/AppData/Local/Temp/cibw-run-5t0hy_xh/cp313-win_amd64/build/venv/Lib/site-packages/scipy_openblas64/include
    lib directory: C:/Users/runneradmin/AppData/Local/Temp/cibw-run-5t0hy_xh/cp313-

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import gymnasium as gym
from collections import deque
from collections import defaultdict
import matplotlib.pyplot as plt
from test_script import QNetwork
from test_script import bar_plot, test_pole_length, test_script

In [3]:
#replay buffer
class ReplayBuffer:
    """
    Replay Buffer to store experience tuples for deep q learning.
    The replay buffer stores experiences from many episodes and randomly samples them during training.
    """
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)
    
    def __len__(self):
        return len(self.buffer)


def select_action(state, policy_net, epsilon, action_dim):
    """
    Select action using epsilon-greedy policy - did it with epsilon-greedy because of Assignent 1
    """
    if random.random() < epsilon:
        return random.randrange(action_dim)
    else:
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0)
            q_values = policy_net(state_tensor)
            return q_values.argmax().item()


def deep_q_learning(epsilon, gamma, alpha, q_network, n_episodes, 
                    pole_lengths=None, env_name='CartPole-v1',
                    batch_size=64, buffer_capacity=50000, 
                    update_target_every=10, epsilon_decay=0.995, 
                    epsilon_min=0.01):
    """
    Deep q learning agent for CartPole-v1 environment with varying pole lengths.
    
    param: epsilon : float - initial exploration rate
    param: gamma : float - discount factor
    param: alpha : float - learning rate
    param: q_network : QNetwork or None - pre-initialized network or None to create new one
    param: n_episodes : int - number of training episodes
    param: pole_lengths : array-like or None - array of pole lengths to train on (default: linspace(0.4, 1.8, 30))
    param: env_name : str - gym environment name
    param: batch_size : int - batch size for training
    param: buffer_capacity : int - replay buffer capacity
    param: update_target_every : int - how often to update target network
    param: epsilon_decay : float - epsilon decay rate per episode
    param: epsilon_min : float - minimum epsilon value

    return: tuple : (policy_net, target_net, episode_returns)
        - trained networks and list of episode rewards
    """

    # initialization of environment
    env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    # initialization of networks if not provided
    if q_network is None:
        policy_net = QNetwork(state_dim, action_dim)
        target_net = QNetwork(state_dim, action_dim)
        target_net.load_state_dict(policy_net.state_dict())
        target_net.eval()
    else:
        policy_net = q_network
        target_net = QNetwork(state_dim, action_dim)
        target_net.load_state_dict(policy_net.state_dict())
        target_net.eval()

    # initialization of optimizer
    optimizer = optim.Adam(policy_net.parameters(), lr=alpha)

    # initialization of replay buffer
    replay_buffer = ReplayBuffer(buffer_capacity)
    
    # pole lengths for training
    if pole_lengths is None:
        pole_lengths = np.linspace(0.4, 1.8, 30)

    # storing episode returns for plotting
    episode_returns = []
    
    # copy of current epsilon value for decay
    epsi = epsilon
    
    # training loop
    for episode in range(n_episodes):
        # randomly select pole length for this episode (we need to figure an experimental setup)
        pole_length = np.random.choice(pole_lengths)
        env.unwrapped.length = pole_length
        
        # reset environment
        state = env.reset()[0]
        episode_reward = 0.0
        
        # epsilon decay
        if epsi > epsilon_min:
            epsi = max(epsilon_min, epsi * epsilon_decay)
        
        # episode loop (1 episode = 1 pole length)
        done = False
        
        while not done:
            # select action
            action = select_action(state, policy_net, epsi, action_dim)
            
            # take step
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # store transition in replay buffer
            replay_buffer.push(state, action, reward, next_state, float(done))

            # deep q learning update (using mini-batch from replay buffer)
            if len(replay_buffer) >= batch_size:
                # sample batch from replay buffer
                batch = replay_buffer.sample(batch_size)
                states, actions, rewards, next_states, dones = zip(*batch)
                
                # convert to tensors 
                states_t = torch.FloatTensor(states)
                actions_t = torch.LongTensor(actions).unsqueeze(1)
                rewards_t = torch.FloatTensor(rewards).unsqueeze(1)
                next_states_t = torch.FloatTensor(next_states)
                dones_t = torch.FloatTensor(dones).unsqueeze(1)

                #get current q values
                current_q = policy_net(states_t).gather(1, actions_t)
                
                # target values
                with torch.no_grad():
                    next_max = target_net(next_states_t).max(1)[0].unsqueeze(1)
                    td_target = rewards_t + gamma * next_max * (1 - dones_t)
                
                # loss calc
                loss = nn.MSELoss()(current_q, td_target)
                
                # backprop and optimize + gradient clipping
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
                optimizer.step()
            
            episode_reward += reward
            state = next_state
        
        
        #update target network periodically
        if episode % update_target_every == 0:
            target_net.load_state_dict(policy_net.state_dict())

        #store episode reward
        episode_returns.append(episode_reward)
        
        #only for seeing the progress
        if episode % 100 == 0:
            avg_reward = np.mean(episode_returns[-100:]) if len(episode_returns) >= 100 else np.mean(episode_returns)
            print(f"Episode {episode}/{n_episodes} | "
                  f"Avg Reward: {avg_reward:.1f} | "
                  f"Epsilon: {epsi:.3f}")
    
    env.close()
    
    return (policy_net, target_net, episode_returns)


In [4]:
class StratifiedReplayBuffer:
    """
    Replay Buffer to store experience tuples for deep q learning.
    The replay buffer stores experiences from many episodes and randomly samples them during training.
    """
    def __init__(self, capacity):
        self.buffer = {}
        self.capacity = capacity
    
    def push(self, label, state, action, reward, next_state, done):
        sample = (state, action, reward, next_state, done)
        if label not in  self.buffer:
            self.buffer[label] = deque()
            num_labels = len(self.buffer)
            new_max_buffer_capacity = self.capacity // num_labels
            remainder = self.capacity % num_labels
            # dynamically adapt deque max cap as we add sampled pole lengths to our superbuffer. 
            for i, existing_label in enumerate(self.buffer.keys()):
                current_maxlen = new_max_buffer_capacity + (1 if i < remainder else 0)        
                old_deque = self.buffer[existing_label]
                self.buffer[existing_label] = deque(old_deque, maxlen=current_maxlen) 


        self.buffer[label].append(sample)

    def sample(self, batch_size):
        if not self.buffer:
            return []
        labels = list(self.buffer.keys()) 
        samples_per_label = batch_size//len(labels)
        remainder = batch_size % len(labels)
        all_samples = []
        # iterate through all labels (pole lengths) to ensure most even distribution in sample (uniform)
        for i, label in enumerate(labels):
            label_buffer = self.buffer[label]
            num_to_sample = samples_per_label + (1 if i < remainder else 0)
            num_to_sample = min(num_to_sample, len(label_buffer))
            batch = random.sample(label_buffer, num_to_sample)
            all_samples.extend(batch)
        
        random.shuffle(all_samples)
        return all_samples
    
    def __len__(self):
        return sum(len(d) for d in self.buffer.values())


def select_action(state, policy_net, epsilon, action_dim):
    """
    Select action using epsilon-greedy policy - did it with epsilon-greedy because of Assignent 1
    """
    if random.random() < epsilon:
        return random.randrange(action_dim)
    else:
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0)
            q_values = policy_net(state_tensor)
            return q_values.argmax().item()


def deep_q_learning_strati_buff(epsilon, gamma, alpha, q_network, n_episodes, 
                    pole_lengths=None, env_name='CartPole-v1',
                    batch_size=64, buffer_capacity=50000, 
                    update_target_every=10, epsilon_decay=0.995, 
                    epsilon_min=0.01):
    """
    Deep q learning agent for CartPole-v1 environment with varying pole lengths.
    
    param: epsilon : float - initial exploration rate
    param: gamma : float - discount factor
    param: alpha : float - learning rate
    param: q_network : QNetwork or None - pre-initialized network or None to create new one
    param: n_episodes : int - number of training episodes
    param: pole_lengths : array-like or None - array of pole lengths to train on (default: linspace(0.4, 1.8, 30))
    param: env_name : str - gym environment name
    param: batch_size : int - batch size for training
    param: buffer_capacity : int - replay buffer capacity
    param: update_target_every : int - how often to update target network
    param: epsilon_decay : float - epsilon decay rate per episode
    param: epsilon_min : float - minimum epsilon value

    return: tuple : (policy_net, target_net, episode_returns)
        - trained networks and list of episode rewards
    """

    # initialization of environment
    env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    # initialization of networks if not provided
    if q_network is None:
        policy_net = QNetwork(state_dim, action_dim)
        target_net = QNetwork(state_dim, action_dim)
        target_net.load_state_dict(policy_net.state_dict())
        target_net.eval()
    else:
        policy_net = q_network
        target_net = QNetwork(state_dim, action_dim)
        target_net.load_state_dict(policy_net.state_dict())
        target_net.eval()

    # initialization of optimizer
    optimizer = optim.Adam(policy_net.parameters(), lr=alpha)

    # initialization of replay buffer
    replay_buffer = StratifiedReplayBuffer(buffer_capacity)
    
    # pole lengths for training
    if pole_lengths is None:
        pole_lengths = np.linspace(0.4, 1.8, 30)

    # storing episode returns for plotting
    episode_returns = []
    
    # copy of current epsilon value for decay
    epsi = epsilon
    
    # training loop
    for episode in range(n_episodes):
        # randomly select pole length for this episode (we need to figure an experimental setup)
        pole_length = np.random.choice(pole_lengths)
        env.unwrapped.length = pole_length
        
        # reset environment
        state = env.reset()[0]
        episode_reward = 0.0
        
        # epsilon decay
        if epsi > epsilon_min:
            epsi = max(epsilon_min, epsi * epsilon_decay)
        
        # episode loop (1 episode = 1 pole length)
        done = False
        
        while not done:
            # select action
            action = select_action(state, policy_net, epsi, action_dim)
            
            # take step
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # store transition in replay buffer
            replay_buffer.push(pole_length, state, action, reward, next_state, float(done))

            # deep q learning update (using mini-batch from replay buffer)
            if len(replay_buffer) >= batch_size:
                # sample batch from replay buffer
                batch = replay_buffer.sample(batch_size)
                states, actions, rewards, next_states, dones = zip(*batch)
                
                # convert to tensors 
                states_t = torch.FloatTensor(states)
                actions_t = torch.LongTensor(actions).unsqueeze(1)
                rewards_t = torch.FloatTensor(rewards).unsqueeze(1)
                next_states_t = torch.FloatTensor(next_states)
                dones_t = torch.FloatTensor(dones).unsqueeze(1)

                #get current q values
                current_q = policy_net(states_t).gather(1, actions_t)
                
                # target values
                with torch.no_grad():
                    next_max = target_net(next_states_t).max(1)[0].unsqueeze(1)
                    td_target = rewards_t + gamma * next_max * (1 - dones_t)
                
                # loss calc
                loss = nn.MSELoss()(current_q, td_target)
                
                # backprop and optimize + gradient clipping
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
                optimizer.step()
            
            episode_reward += reward
            state = next_state
        
        
        #update target network periodically
        if episode % update_target_every == 0:
            target_net.load_state_dict(policy_net.state_dict())

        #store episode reward
        episode_returns.append(episode_reward)
        
        #only for seeing the progress
        if episode % 50 == 0:
            avg_reward = np.mean(episode_returns[-50:]) if len(episode_returns) >= 50 else np.mean(episode_returns)
            print(f"Episode {episode}/{n_episodes} | "
                  f"Avg Reward: {avg_reward:.1f} | "
                  f"Epsilon: {epsi:.3f}")
    
    env.close()
    
    return policy_net, target_net, episode_returns

In [5]:
class AdaptiveCurriculumLearning:
    """ 
    Adaptive curriculum learning offers an object to store the accumulated rewards, 
    performances, difficulty socores, and probabilities for each pole length. 
    We check the last n (LOOK_BACK_WINDOW) rewards a pole has obtained and assign probabilities 
    to sample each length for an episode. The goal is to attack the policy on its weak spots, 
    we prioritize training on weak performing lengths. This should shift performance metrics and keeps 
    our attacks adaptive as we target weakest. 
    param: all_pole_lengths : numpy.ndarray - stores all the pole lengths
    """
    LOOK_BACK_WINDOW = 20 # we only consider the last 20 rewards a pole has gotten (captures more sensitive information)
    P_ADAPTIVE = 0.8 # probability of using adaptive probability distribution or uniform random
    def __init__(self, all_pole_lengths):
        self.all_pole_lengths = all_pole_lengths
        self.rewards = defaultdict(list) # storing pole length episode reward for approach 2 (see 2.2)
        self.performances = {} # keep track of performance metric for each pole length 
        self.difficulty_scores = {} # difficulty_scores
        self.distribution = {} # probability distribution for pole lengths to sample from
        
        # initialize difficulties and probabilities (initially prob is uniform)
        initial_prob = 1.0 / len(all_pole_lengths)
        self.i_p = initial_prob
        for length in all_pole_lengths:
            self.difficulty_scores[length] = 1.0
            self.distribution[length] = initial_prob

    def update_rewards(self, pole_length, reward):
        self.rewards[pole_length].append(reward)

    def update_performances(self, pole_length):
        """
        Here we update the performance of a pole, 
        """
        reward_list = self.rewards[pole_length]

        # if a pole has not been played yet, it will be assigned a performance metric of 0
        if not reward_list:
            metric = 0
        else:
            # here we use a LOOK BACK WINDOW so that we dont use over-stabilized episode rewards
            metric = np.mean(reward_list[-self.LOOK_BACK_WINDOW:])
        self.performances[pole_length] = metric

    def update_difficulties(self):
        """
        Difficulties are inversely proportional to the performance metrics.
        The worse a pole length has performed, the higher the diff score (diff=1 being the worst performing, diff=0 being best).
        These require global updates as the update is relative to the totality of pole lengths.
        Also normalizing + scaling the values down. Metrics can offer quite larger values otherwise. 
        """
        M_max = self.find_max()
        M_min = self.find_min()
        diff_M = M_max - M_min

        # update all difficulty scores with new information
        for pole_length, metric in self.performances.items():
            if diff_M == 0:
                difficulty = 1
            else:
                difficulty = 1 - ((metric-M_min) / diff_M) # if best perform, diff is 0 >>> probability assignment will be 0
            self.difficulty_scores[pole_length] = difficulty
    
    def update_distribution(self):
        """ 
        Distribution is also global, here we require an update proportional to the totality. 
        Normalizing the probabilities (between 0 and 1)
        """
        if not self.difficulty_scores:
            return
        total_difficulty = sum(self.difficulty_scores.values())

        if total_difficulty > 0:
            for pole_length, difficulty in self.difficulty_scores.items():
                self.distribution[pole_length] = difficulty / total_difficulty
        else:
            for pole_length in self.all_pole_lengths:
                self.distribution[pole_length] = self.i_p
    
    def sample_length(self):
        """
        Here select the pole length using either uniform or categorical prob distribution. 
        """
        if random.random() < self.P_ADAPTIVE:
            pole_lengths = list(self.distribution.keys())
            probs = list(self.distribution.values())

            # if our prob dist is empty, we fallback to uniform selection
            if not pole_lengths or sum(probs) == 0:
                return np.random.choice(self.all_pole_lengths)
            
            # selection based on categorical sampling (discrete prob distribution)
            return np.random.choice(a=pole_lengths, p=probs, size=1)[0]
        else:
            return np.random.choice(self.all_pole_lengths)

    def find_max(self):
        """
        Find max performing pole length
        """
        if not self.performances:
            return 0
        return max(self.performances.values())

    def find_min(self):
        """
        Find min performing pole length
        """
        if not self.performances:
            return 0
        return min(self.performances.values())
    
    def calculate_pole_stats(self):
        """
        Nice display function for avg reward of each pole. Not functionally important at all. 
        """
        avg_pole_stats = {}
        
        for pole_length, rewards_list in self.rewards.items():
            # Only process if the list is not empty
            if rewards_list:
                avg_reward = np.mean(rewards_list)
                count = len(rewards_list)
            else:
                avg_reward = 0.0
                count = 0
            avg_pole_stats[pole_length] = {
                "average_reward": avg_reward,
                "episode_count": count
            }
        for p, avg in avg_pole_stats.items():
            print(f"Pole length {p} has avg reward {avg}")
        return avg_pole_stats

class ReplayBuffer:
    """
    Replay Buffer to store experience tuples for deep q learning.
    The replay buffer stores experiences from many episodes and randomly samples them during training.
    """
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)
    
    def __len__(self):
        return len(self.buffer)


def select_action(state, policy_net, epsilon, action_dim):
    """
    Select action using epsilon-greedy policy - did it with epsilon-greedy because of Assignent 1
    """
    if random.random() < epsilon:
        return random.randrange(action_dim)
    else:
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0)
            q_values = policy_net(state_tensor)
            return q_values.argmax().item()


def deep_q_learning_acl(epsilon, gamma, alpha, q_network, n_episodes,
                    uniform_episode_training_cap=300,
                    pole_lengths=None, env_name='CartPole-v1',
                    batch_size=64, buffer_capacity=50000, 
                    update_target_every=10, epsilon_decay=0.995, 
                    epsilon_min=0.01):
    """
    Deep q learning agent for CartPole-v1 environment with varying pole lengths.
    
    param: epsilon : float - initial exploration rate
    param: gamma : float - discount factor
    param: alpha : float - learning rate
    param: q_network : QNetwork or None - pre-initialized network or None to create new one
    param: n_episodes : int - number of training episodes
    param: uniform_episode_training_cap : int - number episodes trained with uniform length selection
    param: p_adaptive : float - probability to select pole length from sample distribution, after uniform_episode_training_cap
    param: lb_window : int - look back window used to compute performance metric, number of recent pole length performances
    param: pole_lengths : array-like or None - array of pole lengths to train on (default: linspace(0.4, 1.8, 30))
    param: env_name : str - gym environment name
    param: batch_size : int - batch size for training
    param: buffer_capacity : int - replay buffer capacity
    param: update_target_every : int - how often to update target network
    param: epsilon_decay : float - epsilon decay rate per episode
    param: epsilon_min : float - minimum epsilon value

    return: tuple : (policy_net, target_net, episode_returns)
        - trained networks and list of episode rewards
    """

    # initialization of environment
    env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    # initialization of networks if not provided
    if q_network is None:
        policy_net = QNetwork(state_dim, action_dim)
        target_net = QNetwork(state_dim, action_dim)
        target_net.load_state_dict(policy_net.state_dict())
        target_net.eval()
    else:
        policy_net = q_network
        target_net = QNetwork(state_dim, action_dim)
        target_net.load_state_dict(policy_net.state_dict())
        target_net.eval()

    # initialization of optimizer
    optimizer = optim.Adam(policy_net.parameters(), lr=alpha)

    # initialization of replay buffer
    replay_buffer = ReplayBuffer(buffer_capacity)
    
    # pole lengths for training
    if pole_lengths is None:
        pole_lengths = np.linspace(0.4, 1.8, 30)

    # storing episode returns for plotting
    episode_returns = []

    # initialize the acl class for adaptive hyperparametre sampling
    acl = AdaptiveCurriculumLearning(pole_lengths)

    # copy of current epsilon value for decay
    epsi = epsilon
    
    # training loop
    for episode in range(n_episodes):
        # if current episode is below the uniform_episode_training_cap we select from a uniform distr
        if episode < uniform_episode_training_cap:
            pole_length = np.random.choice(pole_lengths)
        # else use adaptive probability distrubition 
        else: 
            pole_length = acl.sample_length()

        env.unwrapped.length = pole_length
        
        # reset environment
        state = env.reset()[0]
        episode_reward = 0.0
        
        # epsilon decay
        if epsi > epsilon_min:
            epsi = max(epsilon_min, epsi * epsilon_decay)
        
        # episode loop (1 episode = 1 pole length)
        done = False
        
        while not done:
            # select action
            action = select_action(state, policy_net, epsi, action_dim)
            
            # take step
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # store transition in replay buffer
            replay_buffer.push(state, action, reward, next_state, float(done))

            # deep q learning update (using mini-batch from replay buffer)
            if len(replay_buffer) >= batch_size:
                # sample batch from replay buffer
                batch = replay_buffer.sample(batch_size)
                states, actions, rewards, next_states, dones = zip(*batch)
                
                # convert to tensors 
                states_t = torch.FloatTensor(states)
                actions_t = torch.LongTensor(actions).unsqueeze(1)
                rewards_t = torch.FloatTensor(rewards).unsqueeze(1)
                next_states_t = torch.FloatTensor(next_states)
                dones_t = torch.FloatTensor(dones).unsqueeze(1)

                #get current q values
                current_q = policy_net(states_t).gather(1, actions_t)
                
                # target values
                with torch.no_grad():
                    next_max = target_net(next_states_t).max(1)[0].unsqueeze(1)
                    td_target = rewards_t + gamma * next_max * (1 - dones_t)
                
                # loss calc
                loss = nn.MSELoss()(current_q, td_target)
                
                # backprop and optimize + gradient clipping
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
                optimizer.step()
            
            episode_reward += reward
            state = next_state
        
        
        #update target network periodically
        if episode % update_target_every == 0:
            target_net.load_state_dict(policy_net.state_dict())

        #store episode reward
        episode_returns.append(episode_reward)

        # update acl for rewards, performance metrics, scores, and probs
        # difficulty scores and probabilities are global updates, can be seen in the respective update methods
        acl.update_rewards(pole_length, episode_reward)
        if episode >= uniform_episode_training_cap:
            acl.update_performances(pole_length)
            acl.update_difficulties()
            acl.update_distribution()

        #only for seeing the progress
        if episode % 50 == 0:
            avg_reward = np.mean(episode_returns[-100:]) if len(episode_returns) >= 100 else np.mean(episode_returns)
            print(f"Episode {episode}/{n_episodes} | "
                  f"Avg Reward: {avg_reward:.1f} | "
                  f"Epsilon: {epsi:.3f} | "
                  f"Probability Distribution{acl.distribution} | \n" 
                  f"Pole Length Performances{acl.performances} | \n"
                  )
            
    
    env.close()
    
    return policy_net, target_net, episode_returns, acl

In [6]:
class ExplorationDiversityCurriculum:
    """ 
    Exploration Diversity Curriculum prioritizes pole lengths that have been visited LEAST.
    This ensures balanced coverage of the pole length space and prevents over-training
    on familiar lengths while neglecting others.
    
    basically saying "Explore what you haven't seen"
    
    param: all_pole_lengths : numpy.ndarray - stores all the pole lengths
    """
    
    P_ADAPTIVE = 0.8  #probability of using adaptive (exploration-based) vs uniform
    MIN_VISITS_THRESHOLD = 10  #minimum visits before a length is considered "explored"
    
    def __init__(self, all_pole_lengths):
        self.all_pole_lengths = all_pole_lengths
        self.visit_counts = defaultdict(int)  # track how many times each length was visited
        self.rewards = defaultdict(list) 
        self.rarity_scores = {}  # how "rare" each pole length is (inverse of visits)
        self.distribution = {}  # probability distribution for sampling
        
        #initialize with uniform distribution
        initial_prob = 1.0 / len(all_pole_lengths)
        self.initial_prob = initial_prob
        for length in all_pole_lengths:
            self.rarity_scores[length] = 1.0
            self.distribution[length] = initial_prob
    
    def update_visit(self, pole_length):
        """
        Increment visit counter for pole length
        """
        self.visit_counts[pole_length] += 1
    
    def update_rewards(self, pole_length, reward):
        """
        Track rewards (for analysis purposes)
        """
        self.rewards[pole_length].append(reward)
    
    def update_rarity_scores(self):
        """
        Rarity scores are INVERSELY proportional to visit counts.
        Least visited = highest rarity = highest sampling probability
        
        This creates an exploration bonus for under-explored pole lengths.
        """
        if not self.visit_counts:
            return
        
        max_visits = max(self.visit_counts.values())
        min_visits = min(self.visit_counts.values())
        diff_visits = max_visits - min_visits
        
        for pole_length in self.all_pole_lengths:
            visits = self.visit_counts.get(pole_length, 0)
            
            if diff_visits == 0:
                #all lengths visited equally - uniform rarity
                rarity = 1.0
            else:
                #inverse normalization: least visited = highest rarity (closer to 1)
                #most visited = lowest rarity (closer to 0)
                rarity = 1.0 - ((visits - min_visits) / diff_visits)
            
            #add bonus for completely unvisited lengths
            if visits == 0:
                rarity = 2.0  #double rarity - ultra rare
            
            self.rarity_scores[pole_length] = rarity
    
    def update_distribution(self):
        """
        Update sampling distribution based on rarity scores.
        Normalize probabilities to sum to 1.
        """
        if not self.rarity_scores:
            return
        
        total_rarity = sum(self.rarity_scores.values())
        
        if total_rarity > 0:
            for pole_length, rarity in self.rarity_scores.items():
                self.distribution[pole_length] = rarity / total_rarity
        else:
            #fallback to uniform
            for pole_length in self.all_pole_lengths:
                self.distribution[pole_length] = self.initial_prob
    
    def sample_length(self):
        """
        Sample pole length using either:
        - Adaptive (exploration-based) distribution (P_ADAPTIVE probability)
        - Uniform random (1 - P_ADAPTIVE probability)
        """
        if random.random() < self.P_ADAPTIVE:
            pole_lengths = list(self.distribution.keys())
            probs = list(self.distribution.values())
            
            #fallback to uniform if distribution is invalid
            if not pole_lengths or sum(probs) == 0:
                return np.random.choice(self.all_pole_lengths)
            
            #sample based on rarity (exploration priority)
            return np.random.choice(a=pole_lengths, p=probs, size=1)[0]
        else:
            return np.random.choice(self.all_pole_lengths)
    
    def get_exploration_stats(self):
        """
        Return statistics about exploration coverage
        """
        total_visits = sum(self.visit_counts.values())
        unvisited = [l for l in self.all_pole_lengths if self.visit_counts[l] == 0]
        under_explored = [l for l in self.all_pole_lengths 
                         if 0 < self.visit_counts[l] < self.MIN_VISITS_THRESHOLD]
        
        return {
            'total_visits': total_visits,
            'unvisited_count': len(unvisited),
            'under_explored_count': len(under_explored),
            'visit_counts': dict(self.visit_counts),
            'min_visits': min(self.visit_counts.values()) if self.visit_counts else 0,
            'max_visits': max(self.visit_counts.values()) if self.visit_counts else 0,
        }
    
    def calculate_pole_stats(self):
        """
        Display statistics for each pole length
        """
        stats = {}
        
        for pole_length in self.all_pole_lengths:
            visits = self.visit_counts.get(pole_length, 0)
            rewards_list = self.rewards.get(pole_length, [])
            
            if rewards_list:
                avg_reward = np.mean(rewards_list)
            else:
                avg_reward = 0.0
            
            stats[pole_length] = {
                "visits": visits,
                "average_reward": avg_reward,
                "rarity_score": self.rarity_scores.get(pole_length, 1.0),
                "sampling_prob": self.distribution.get(pole_length, self.initial_prob)
            }
        
        return stats


def select_action(state, policy_net, epsilon, action_dim):
    """
    Select action using epsilon-greedy policy
    """
    if random.random() < epsilon:
        return random.randrange(action_dim)
    else:
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0)
            q_values = policy_net(state_tensor)
            return q_values.argmax().item()


def deep_q_learning_exploration_diversity(
        epsilon, gamma, alpha, q_network, n_episodes,
        uniform_episode_training_cap=300,
        pole_lengths=None, env_name='CartPole-v1',
        batch_size=64, buffer_capacity=50000, 
        update_target_every=10, epsilon_decay=0.995, 
        epsilon_min=0.01):
    """
    Deep q learning with Exploration Diversity Curriculum
    
    This strategy prioritizes training on LEAST-VISITED pole lengths
    
    param: epsilon : float - initial exploration rate
    param: gamma : float - discount factor
    param: alpha : float - learning rate
    param: q_network : QNetwork or None - pre-initialized network or None to create new one
    param: n_episodes : int - number of training episodes
    param: uniform_episode_training_cap : int - number episodes trained with uniform length selection
    param: p_adaptive : float - probability to select pole length from sample distribution, after uniform_episode_training_cap
    param: lb_window : int - look back window used to compute performance metric, number of recent pole length performances
    param: pole_lengths : array-like or None - array of pole lengths to train on (default: linspace(0.4, 1.8, 30))
    param: env_name : str - gym environment name
    param: batch_size : int - batch size for training
    param: buffer_capacity : int - replay buffer capacity
    param: update_target_every : int - how often to update target network
    param: epsilon_decay : float - epsilon decay rate per episode
    param: epsilon_min : float - minimum epsilon value
    
    return: tuple : (policy_net, target_net, episode_returns, exploration_diversity_curriculum)
        - trained networks and list of episode rewards
    """
    
    #initialize environment
    env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n
    
    #initialize networks
    if q_network is None:
        policy_net = QNetwork(state_dim, action_dim)
        target_net = QNetwork(state_dim, action_dim)
        target_net.load_state_dict(policy_net.state_dict())
        target_net.eval()
    else:
        policy_net = q_network
        target_net = QNetwork(state_dim, action_dim)
        target_net.load_state_dict(policy_net.state_dict())
        target_net.eval()
    
    #initialize optimizer
    optimizer = optim.Adam(policy_net.parameters(), lr=alpha)
    
    #initialize replay buffer
    replay_buffer = ReplayBuffer(buffer_capacity)
    
    #pole lengths for training
    if pole_lengths is None:
        pole_lengths = np.linspace(0.4, 1.8, 30)
    
    #storing episode returns
    episode_returns = []
    
    #initialize Exploration Diversity Curriculum
    exploration_diversity_curriculum = ExplorationDiversityCurriculum(pole_lengths)
    
    #epsilon copy
    epsi = epsilon
    
    #training loop
    for episode in range(n_episodes):
        #phase 1: uniform sampling for initial exploration
        if episode < uniform_episode_training_cap:
            pole_length = np.random.choice(pole_lengths)
        #phase 2: prioritize least-visited lengths
        else:
            pole_length = exploration_diversity_curriculum.sample_length()
        
        env.unwrapped.length = pole_length
        
        #track that we visited this length
        exploration_diversity_curriculum.update_visit(pole_length)
        
        #reset environment
        state = env.reset()[0]
        episode_reward = 0.0
        
        #epsilon decay
        if epsi > epsilon_min:
            epsi = max(epsilon_min, epsi * epsilon_decay)
        
        #episode loop
        done = False
        
        while not done:
            #select action
            action = select_action(state, policy_net, epsi, action_dim)
            
            #take step
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            #store transition
            replay_buffer.push(state, action, reward, next_state, float(done))
            
            #training step
            if len(replay_buffer) >= batch_size:
                batch = replay_buffer.sample(batch_size)
                states, actions, rewards, next_states, dones = zip(*batch)
                
                states_t = torch.FloatTensor(states)
                actions_t = torch.LongTensor(actions).unsqueeze(1)
                rewards_t = torch.FloatTensor(rewards).unsqueeze(1)
                next_states_t = torch.FloatTensor(next_states)
                dones_t = torch.FloatTensor(dones).unsqueeze(1)
                
                current_q = policy_net(states_t).gather(1, actions_t)
                
                with torch.no_grad():
                    next_max = target_net(next_states_t).max(1)[0].unsqueeze(1)
                    td_target = rewards_t + gamma * next_max * (1 - dones_t)
                
                loss = nn.MSELoss()(current_q, td_target)
                
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
                optimizer.step()
            
            episode_reward += reward
            state = next_state
        
        #update target network periodically
        if episode % update_target_every == 0:
            target_net.load_state_dict(policy_net.state_dict())
        
        #store episode return
        episode_returns.append(episode_reward)
        
        #update curriculum: track rewards and update rarity-based distribution
        exploration_diversity_curriculum.update_rewards(pole_length, episode_reward)
        
        if episode >= uniform_episode_training_cap:
            exploration_diversity_curriculum.update_rarity_scores()
            exploration_diversity_curriculum.update_distribution()

        #progress logging
        if episode % 50 == 0:
            avg_reward = np.mean(episode_returns[-100:]) if len(episode_returns) >= 100 else np.mean(episode_returns)
            exploration_stats = exploration_diversity_curriculum.get_exploration_stats()

            print(f"Episode {episode}/{n_episodes} | "
                  f"Avg Reward: {avg_reward:.1f} | "
                  f"Epsilon: {epsi:.3f}")
            print(f"  Exploration Stats: Unvisited={exploration_stats['unvisited_count']}, "
                  f"Min visits={exploration_stats['min_visits']}, "
                  f"Max visits={exploration_stats['max_visits']}")
            
            #show top 3 most likely to be sampled (least visited)
            top_3_rare = sorted(exploration_diversity_curriculum.distribution.items(), key=lambda x: x[1], reverse=True)[:3]
            print(f"  Prioritizing (least visited): {[f'{l:.2f}({p:.3f})' for l, p in top_3_rare]}")
    
    env.close()

    #am returning also the policy in case we want to save it to test later 
    return policy_net, target_net, episode_returns, exploration_diversity_curriculum

In [7]:
# import os
# import sys

# # ================================
# # CRITICAL: SET THESE FIRST!
# # Must be set BEFORE importing numpy, torch, or anything else
# # ================================
# os.environ["OMP_NUM_THREADS"] = "1"
# os.environ["MKL_NUM_THREADS"] = "1"  
# os.environ["NUMEXPR_NUM_THREADS"] = "1"
# os.environ["OPENBLAS_NUM_THREADS"] = "1"
# os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

# # NOW import libraries
# import multiprocessing as mp
# import torch
# import numpy as np
# import optuna
# import logging
# from datetime import datetime
# import psutil
# import json

# # Force single-threaded PyTorch
# torch.set_num_threads(1)

# # Verify it worked
# print("="*80)
# print("THREAD CONFIGURATION CHECK")
# print("="*80)
# print(f"PyTorch threads: {torch.get_num_threads()} (should be 1)")
# print(f"OMP_NUM_THREADS: {os.environ.get('OMP_NUM_THREADS')}")
# print(f"MKL_NUM_THREADS: {os.environ.get('MKL_NUM_THREADS')}")
# print(f"OPENBLAS_NUM_THREADS: {os.environ.get('OPENBLAS_NUM_THREADS')}")
# print("="*80 + "\n")

# # Setup logging
# def setup_logging():
#     log_file = f'training_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
#     logging.basicConfig(
#         level=logging.INFO,
#         format='%(asctime)s [%(processName)-12s] %(message)s',
#         datefmt='%H:%M:%S',
#         handlers=[
#             logging.FileHandler(log_file),
#             logging.StreamHandler()
#         ],
#         force=True
#     )
#     return log_file

# def objective_baseline(trial):
#     trial_start = datetime.now()
#     trial_num = trial.number + 1
    
#     logging.info(f"🔵 BASELINE Trial {trial_num}/10 STARTED")
    
#     gamma = trial.suggest_float('gamma', 0.95, 0.999)
#     alpha = trial.suggest_float('alpha', 1e-4, 1e-2, log=True)
#     batch_size = trial.suggest_categorical('batch_size', [32, 64])
#     buffer_capacity = trial.suggest_categorical('buffer_capacity', [5000, 10000, 15000])
#     update_target_every = trial.suggest_int('update_target_every', 3, 20)
#     epsilon_decay = trial.suggest_float('epsilon_decay', 0.995, 0.9995)
#     epsilon_min = trial.suggest_float('epsilon_min', 0.01, 0.1)
    
#     logging.info(f"   Params: γ={gamma:.3f}, α={alpha:.2e}, batch={batch_size}, buffer={buffer_capacity}")
    
#     try:
#         policy_net, target_net, returns = deep_q_learning(
#             epsilon=1.0,
#             gamma=gamma,
#             alpha=alpha,
#             q_network=None,
#             n_episodes=2500,
#             pole_lengths=np.linspace(0.4, 1.8, 15),
#             env_name='CartPole-v1',
#             batch_size=batch_size,
#             buffer_capacity=buffer_capacity,
#             update_target_every=update_target_every,
#             epsilon_decay=epsilon_decay,
#             epsilon_min=epsilon_min
#         )
        
#         score = np.mean(returns[-500:])
#         elapsed_min = (datetime.now() - trial_start).total_seconds() / 60
        
#         logging.info(f"✅ BASELINE Trial {trial_num}/10 DONE | Score: {score:.2f} | Time: {elapsed_min:.1f}min")
#         return score
        
#     except Exception as e:
#         logging.error(f"❌ BASELINE Trial {trial_num}/10 FAILED: {e}")
#         return float('-inf')

# def objective_strati_buff(trial):
#     trial_start = datetime.now()
#     trial_num = trial.number + 1
    
#     logging.info(f"🟢 STRATI-BUFF Trial {trial_num}/10 STARTED")
    
#     gamma = trial.suggest_float('gamma', 0.95, 0.999)
#     alpha = trial.suggest_float('alpha', 1e-4, 1e-2, log=True)
#     batch_size = trial.suggest_categorical('batch_size', [32, 64])
#     buffer_capacity = trial.suggest_categorical('buffer_capacity', [5000, 10000])
#     update_target_every = trial.suggest_int('update_target_every', 3, 10)
#     epsilon_decay = trial.suggest_float('epsilon_decay', 0.995, 0.9995)
#     epsilon_min = trial.suggest_float('epsilon_min', 0.01, 0.1)
    
#     logging.info(f"   Params: γ={gamma:.3f}, α={alpha:.2e}, batch={batch_size}, buffer={buffer_capacity}")
    
#     try:
#         policy_net, target_net, returns = deep_q_learning_strati_buff(
#             epsilon=1.0,
#             gamma=gamma,
#             alpha=alpha,
#             q_network=None,
#             n_episodes=2500,
#             pole_lengths=np.linspace(0.4, 1.8, 15),
#             env_name='CartPole-v1',
#             batch_size=batch_size,
#             buffer_capacity=buffer_capacity,
#             update_target_every=update_target_every,
#             epsilon_decay=epsilon_decay,
#             epsilon_min=epsilon_min
#         )
        
#         score = np.mean(returns[-500:])
#         elapsed_min = (datetime.now() - trial_start).total_seconds() / 60
        
#         logging.info(f"✅ STRATI-BUFF Trial {trial_num}/10 DONE | Score: {score:.2f} | Time: {elapsed_min:.1f}min")
#         return score
        
#     except Exception as e:
#         logging.error(f"❌ STRATI-BUFF Trial {trial_num}/10 FAILED: {e}")
#         return float('-inf')

# def objective_acl(trial):
#     trial_start = datetime.now()
#     trial_num = trial.number + 1
    
#     logging.info(f"🟡 ACL Trial {trial_num}/10 STARTED")
    
#     gamma = trial.suggest_float('gamma', 0.95, 0.999)
#     alpha = trial.suggest_float('alpha', 1e-4, 1e-2, log=True)
#     batch_size = trial.suggest_categorical('batch_size', [32, 64])
#     buffer_capacity = trial.suggest_categorical('buffer_capacity', [5000, 10000])
#     update_target_every = trial.suggest_int('update_target_every', 3, 10)
#     epsilon_decay = trial.suggest_float('epsilon_decay', 0.995, 0.9995)
#     epsilon_min = trial.suggest_float('epsilon_min', 0.01, 0.1)
#     uniform_episode_training_cap = trial.suggest_int('uniform_episode_training_cap', 1500, 2000)
    
#     logging.info(f"   Params: γ={gamma:.3f}, α={alpha:.2e}, batch={batch_size}, cap={uniform_episode_training_cap}")
    
#     try:
#         policy_net, target_net, returns, acl = deep_q_learning_acl(
#             epsilon=1.0,
#             gamma=gamma,
#             alpha=alpha,
#             q_network=None,
#             n_episodes=2500,
#             uniform_episode_training_cap=uniform_episode_training_cap,
#             pole_lengths=np.linspace(0.4, 1.8, 15),
#             env_name='CartPole-v1',
#             batch_size=batch_size,
#             buffer_capacity=buffer_capacity,
#             update_target_every=update_target_every,
#             epsilon_decay=epsilon_decay,
#             epsilon_min=epsilon_min
#         )
        
#         score = np.mean(returns[-500:])
#         elapsed_min = (datetime.now() - trial_start).total_seconds() / 60
        
#         logging.info(f"✅ ACL Trial {trial_num}/10 DONE | Score: {score:.2f} | Time: {elapsed_min:.1f}min")
#         return score
        
#     except Exception as e:
#         logging.error(f"❌ ACL Trial {trial_num}/10 FAILED: {e}")
#         return float('-inf')

# def objective_exploration_diversity(trial):
#     trial_start = datetime.now()
#     trial_num = trial.number + 1
    
#     logging.info(f"🔴 EXPLORATION Trial {trial_num}/10 STARTED")
    
#     gamma = trial.suggest_float('gamma', 0.95, 0.999)
#     alpha = trial.suggest_float('alpha', 1e-4, 1e-2, log=True)
#     batch_size = trial.suggest_categorical('batch_size', [32, 64])
#     buffer_capacity = trial.suggest_categorical('buffer_capacity', [5000, 10000])
#     update_target_every = trial.suggest_int('update_target_every', 3, 10)
#     epsilon_decay = trial.suggest_float('epsilon_decay', 0.995, 0.9995)
#     epsilon_min = trial.suggest_float('epsilon_min', 0.01, 0.1)
#     uniform_episode_training_cap = trial.suggest_int('uniform_episode_training_cap', 1500, 2000)
    
#     logging.info(f"   Params: γ={gamma:.3f}, α={alpha:.2e}, batch={batch_size}, cap={uniform_episode_training_cap}")
    
#     try:
#         policy_net, target_net, returns, curriculum = deep_q_learning_exploration_diversity(
#             epsilon=1.0,
#             gamma=gamma,
#             alpha=alpha,
#             q_network=None,
#             n_episodes=2500,
#             uniform_episode_training_cap=uniform_episode_training_cap,
#             pole_lengths=np.linspace(0.4, 1.8, 15),
#             env_name='CartPole-v1',
#             batch_size=batch_size,
#             buffer_capacity=buffer_capacity,
#             update_target_every=update_target_every,
#             epsilon_decay=epsilon_decay,
#             epsilon_min=epsilon_min
#         )
        
#         score = np.mean(returns[-500:])
#         elapsed_min = (datetime.now() - trial_start).total_seconds() / 60
        
#         logging.info(f"✅ EXPLORATION Trial {trial_num}/10 DONE | Score: {score:.2f} | Time: {elapsed_min:.1f}min")
#         return score
        
#     except Exception as e:
#         logging.error(f"❌ EXPLORATION Trial {trial_num}/10 FAILED: {e}")
#         return float('-inf')

# def optimize_model(model_name, objective_func, n_trials=10):
#     start_time = datetime.now()
#     logging.info(f"\n{'='*60}")
#     logging.info(f"🚀 {model_name.upper()} - OPTIMIZATION STARTED")
#     logging.info(f"{'='*60}\n")
    
#     study = optuna.create_study(
#         direction='maximize',
#         pruner=optuna.pruners.MedianPruner(
#             n_startup_trials=3,
#             n_warmup_steps=500,
#             interval_steps=100
#         ),
#         study_name=model_name
#     )
    
#     study.optimize(objective_func, n_trials=n_trials, n_jobs=1, show_progress_bar=False)
    
#     elapsed_hours = (datetime.now() - start_time).total_seconds() / 3600
    
#     logging.info(f"\n{'='*60}")
#     logging.info(f"🏁 {model_name.upper()} - OPTIMIZATION COMPLETE")
#     logging.info(f"   Best Score: {study.best_value:.2f}")
#     logging.info(f"   Total Time: {elapsed_hours:.2f} hours")
#     logging.info(f"   Best Params: {study.best_params}")
#     logging.info(f"{'='*60}\n")
    
#     results = {
#         'model': model_name,
#         'best_value': study.best_value,
#         'best_params': study.best_params,
#         'total_time_hours': elapsed_hours,
#         'timestamp': datetime.now().isoformat()
#     }
    
#     with open(f'{model_name}_best_params.json', 'w') as f:
#         json.dump(results, f, indent=2)
    
#     return results

# def parallel_hyperparameter_tuning(n_trials_per_model=10):
#     logging.info("="*80)
#     logging.info("STARTING PARALLEL HYPERPARAMETER OPTIMIZATION")
#     logging.info(f"Trials per model: {n_trials_per_model}")
#     logging.info(f"Episodes per trial: 2500")
#     logging.info("="*80)
    
#     models = [
#         ('baseline', objective_baseline),
#         ('strati_buff', objective_strati_buff),
#         ('acl', objective_acl),
#         ('exploration_diversity', objective_exploration_diversity)
#     ]
    
#     # Run all optimizations in parallel (4 processes on 4 physical cores)
#     with mp.Pool(processes=4) as pool:
#         results = pool.starmap(
#             optimize_model,
#             [(name, obj, n_trials_per_model) for name, obj in models]
#         )
    
#     logging.info("\n" + "="*80)
#     logging.info("ALL OPTIMIZATIONS COMPLETED")
#     logging.info("="*80)
    
#     logging.info("\nSUMMARY:")
#     for result in results:
#         logging.info(f"\n{result['model'].upper()}:")
#         logging.info(f"  Best Score: {result['best_value']:.2f}")
#         logging.info(f"  Time: {result['total_time_hours']:.2f}h")
#         logging.info(f"  Best Params:")
#         for key, value in result['best_params'].items():
#             logging.info(f"    {key}: {value}")
    
#     return results

# def train_with_best_params(model_name):
#     """Train a model with its best hyperparameters"""
#     # Load best params
#     with open(f'{model_name}_best_params.json', 'r') as f:
#         data = json.load(f)
    
#     params = data['best_params']
#     logging.info(f"\nTraining {model_name} with best parameters...")
    
#     if model_name == 'baseline':
#         policy_net, target_net, returns = deep_q_learning(
#             epsilon=1.0,
#             gamma=params['gamma'],
#             alpha=params['alpha'],
#             q_network=None,
#             n_episodes=2500,
#             pole_lengths=np.linspace(0.4, 1.8, 15),
#             env_name='CartPole-v1',
#             batch_size=params['batch_size'],
#             buffer_capacity=params['buffer_capacity'],
#             update_target_every=params['update_target_every'],
#             epsilon_decay=params['epsilon_decay'],
#             epsilon_min=params['epsilon_min']
#         )
#     elif model_name == 'strati_buff':
#         policy_net, target_net, returns = deep_q_learning_strati_buff(
#             epsilon=1.0,
#             gamma=params['gamma'],
#             alpha=params['alpha'],
#             q_network=None,
#             n_episodes=2500,
#             pole_lengths=np.linspace(0.4, 1.8, 15),
#             env_name='CartPole-v1',
#             batch_size=params['batch_size'],
#             buffer_capacity=params['buffer_capacity'],
#             update_target_every=params['update_target_every'],
#             epsilon_decay=params['epsilon_decay'],
#             epsilon_min=params['epsilon_min']
#         )
#     elif model_name == 'acl':
#         policy_net, target_net, returns, _ = deep_q_learning_acl(
#             epsilon=1.0,
#             gamma=params['gamma'],
#             alpha=params['alpha'],
#             q_network=None,
#             n_episodes=2500,
#             uniform_episode_training_cap=params['uniform_episode_training_cap'],
#             pole_lengths=np.linspace(0.4, 1.8, 15),
#             env_name='CartPole-v1',
#             batch_size=params['batch_size'],
#             buffer_capacity=params['buffer_capacity'],
#             update_target_every=params['update_target_every'],
#             epsilon_decay=params['epsilon_decay'],
#             epsilon_min=params['epsilon_min']
#         )
#     elif model_name == 'exploration_diversity':
#         policy_net, target_net, returns, _ = deep_q_learning_exploration_diversity(
#             epsilon=1.0,
#             gamma=params['gamma'],
#             alpha=params['alpha'],
#             q_network=None,
#             n_episodes=2500,
#             uniform_episode_training_cap=params['uniform_episode_training_cap'],
#             pole_lengths=np.linspace(0.4, 1.8, 15),
#             env_name='CartPole-v1',
#             batch_size=params['batch_size'],
#             buffer_capacity=params['buffer_capacity'],
#             update_target_every=params['update_target_every'],
#             epsilon_decay=params['epsilon_decay'],
#             epsilon_min=params['epsilon_min']
#         )
    
#     torch.save(policy_net.state_dict(), f"{model_name}_optimized_policy.pth")
#     logging.info(f"Saved optimized model to {model_name}_optimized_policy.pth")
    
#     return policy_net, returns

In [8]:

# mp.set_start_method('spawn', force=True)
    
# # Setup logging
# log_file = setup_logging()
# logging.info(f"Logging to: {log_file}\n")

# # Run optimization
# results = parallel_hyperparameter_tuning(n_trials_per_model=10)

# logging.info("\n🎉 ALL DONE!")

# # STEP 2: Train final models with best hyperparameters (parallel)
# logging.info("\n" + "=" * 80)
# logging.info("TRAINING FINAL MODELS WITH BEST HYPERPARAMETERS")
# logging.info("=" * 80)
    
# with mp.Pool(processes=4) as pool:
#         final_models = pool.map(
#             train_with_best_params,
#             ['baseline', 'strati_buff', 'acl', 'exploration_diversity']
#         )
    
# logging.info("\n" + "=" * 80)
# logging.info("COMPLETE! All models optimized and trained.")

In [9]:
# import os
# os.environ["OMP_NUM_THREADS"] = "1"
# os.environ["MKL_NUM_THREADS"] = "1"
# os.environ["OPENBLAS_NUM_THREADS"] = "1"

# import multiprocessing as mp
# import time

# def simple_test(n):
#     print(f"Worker {n} started!")
#     time.sleep(2)
#     print(f"Worker {n} finished!")
#     return n * 2

# mp.set_start_method('spawn', force=True)
    
# print("Testing multiprocessing...")
# with mp.Pool(4) as pool:
#     results = pool.map(simple_test, [1, 2, 3, 4])
    
# print(f"Results: {results}")
# print("Multiprocessing works!")

In [10]:
import os

# CRITICAL: Set BEFORE importing anything
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

import threading
import torch
import numpy as np
import optuna
import logging
from datetime import datetime
import json
from queue import Queue

# Force single-threaded PyTorch
torch.set_num_threads(1)

print("="*80)
print("THREAD CONFIGURATION")
print("="*80)
print(f"PyTorch threads: {torch.get_num_threads()}")
print(f"OMP_NUM_THREADS: {os.environ.get('OMP_NUM_THREADS')}")
print("="*80 + "\n")

# Setup logging
def setup_logging():
    log_file = f'training_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s [%(threadName)-15s] %(message)s',
        datefmt='%H:%M:%S',
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ],
        force=True
    )
    return log_file

def objective_baseline(trial):
    trial_start = datetime.now()
    trial_num = trial.number + 1
    
    logging.info(f"BASELINE Trial {trial_num}/10 STARTED")
    
    gamma = trial.suggest_float('gamma', 0.95, 0.999)
    alpha = trial.suggest_float('alpha', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [32, 64])
    buffer_capacity = trial.suggest_categorical('buffer_capacity', [5000, 10000, 15000])
    update_target_every = trial.suggest_int('update_target_every', 3, 20)
    epsilon_decay = trial.suggest_float('epsilon_decay', 0.995, 0.9995)
    epsilon_min = trial.suggest_float('epsilon_min', 0.01, 0.1)
    
    logging.info(f"   Params: γ={gamma:.3f}, α={alpha:.2e}, batch={batch_size}, buffer={buffer_capacity}")
    
    try:
        policy_net, target_net, returns = deep_q_learning(
            epsilon=1.0,
            gamma=gamma,
            alpha=alpha,
            q_network=None,
            n_episodes=2500,
            pole_lengths=np.linspace(0.4, 1.8, 15),
            env_name='CartPole-v1',
            batch_size=batch_size,
            buffer_capacity=buffer_capacity,
            update_target_every=update_target_every,
            epsilon_decay=epsilon_decay,
            epsilon_min=epsilon_min
        )
        
        score = np.mean(returns[-500:])
        elapsed_min = (datetime.now() - trial_start).total_seconds() / 60
        
        logging.info(f"BASELINE Trial {trial_num}/10 DONE | Score: {score:.2f} | Time: {elapsed_min:.1f}min")
        return score
        
    except Exception as e:
        logging.error(f"BASELINE Trial {trial_num}/10 FAILED: {e}")
        return float('-inf')

def objective_strati_buff(trial):
    trial_start = datetime.now()
    trial_num = trial.number + 1

    logging.info(f"STRATI-BUFF Trial {trial_num}/10 STARTED")

    gamma = trial.suggest_float('gamma', 0.95, 0.999)
    alpha = trial.suggest_float('alpha', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [32, 64])
    buffer_capacity = trial.suggest_categorical('buffer_capacity', [5000, 10000, 15000])
    update_target_every = trial.suggest_int('update_target_every', 3, 20)
    epsilon_decay = trial.suggest_float('epsilon_decay', 0.995, 0.9995)
    epsilon_min = trial.suggest_float('epsilon_min', 0.01, 0.1)
    
    logging.info(f"   Params: γ={gamma:.3f}, α={alpha:.2e}, batch={batch_size}, buffer={buffer_capacity}")
    
    try:
        policy_net, target_net, returns = deep_q_learning_strati_buff(
            epsilon=1.0,
            gamma=gamma,
            alpha=alpha,
            q_network=None,
            n_episodes=2500,
            pole_lengths=np.linspace(0.4, 1.8, 15),
            env_name='CartPole-v1',
            batch_size=batch_size,
            buffer_capacity=buffer_capacity,
            update_target_every=update_target_every,
            epsilon_decay=epsilon_decay,
            epsilon_min=epsilon_min
        )
        
        score = np.mean(returns[-500:])
        elapsed_min = (datetime.now() - trial_start).total_seconds() / 60
        
        logging.info(f"STRATI-BUFF Trial {trial_num}/10 DONE | Score: {score:.2f} | Time: {elapsed_min:.1f}min")
        return score
        
    except Exception as e:
        logging.error(f"STRATI-BUFF Trial {trial_num}/10 FAILED: {e}")
        return float('-inf')

def objective_acl(trial):
    trial_start = datetime.now()
    trial_num = trial.number + 1
    
    logging.info(f"ACL Trial {trial_num}/10 STARTED")
    
    gamma = trial.suggest_float('gamma', 0.95, 0.999)
    alpha = trial.suggest_float('alpha', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [32, 64])
    buffer_capacity = trial.suggest_categorical('buffer_capacity', [5000, 10000, 15000])
    update_target_every = trial.suggest_int('update_target_every', 3, 20)
    epsilon_decay = trial.suggest_float('epsilon_decay', 0.995, 0.9995)
    epsilon_min = trial.suggest_float('epsilon_min', 0.01, 0.1)
    uniform_episode_training_cap = trial.suggest_int('uniform_episode_training_cap', 1500, 2000)
    
    logging.info(f"   Params: γ={gamma:.3f}, α={alpha:.2e}, batch={batch_size}, cap={uniform_episode_training_cap}")
    
    try:
        policy_net, target_net, returns, acl = deep_q_learning_acl(
            epsilon=1.0,
            gamma=gamma,
            alpha=alpha,
            q_network=None,
            n_episodes=2500,
            uniform_episode_training_cap=uniform_episode_training_cap,
            pole_lengths=np.linspace(0.4, 1.8, 15),
            env_name='CartPole-v1',
            batch_size=batch_size,
            buffer_capacity=buffer_capacity,
            update_target_every=update_target_every,
            epsilon_decay=epsilon_decay,
            epsilon_min=epsilon_min
        )
        
        score = np.mean(returns[-500:])
        elapsed_min = (datetime.now() - trial_start).total_seconds() / 60
        
        logging.info(f"ACL Trial {trial_num}/10 DONE | Score: {score:.2f} | Time: {elapsed_min:.1f}min")
        return score
        
    except Exception as e:
        logging.error(f"ACL Trial {trial_num}/10 FAILED: {e}")
        return float('-inf')

def objective_exploration_diversity(trial):
    trial_start = datetime.now()
    trial_num = trial.number + 1
    
    logging.info(f"EXPLORATION Trial {trial_num}/10 STARTED")
    
    gamma = trial.suggest_float('gamma', 0.95, 0.999)
    alpha = trial.suggest_float('alpha', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [32, 64])
    buffer_capacity = trial.suggest_categorical('buffer_capacity', [5000, 10000, 15000])
    update_target_every = trial.suggest_int('update_target_every', 3, 20)
    epsilon_decay = trial.suggest_float('epsilon_decay', 0.995, 0.9995)
    epsilon_min = trial.suggest_float('epsilon_min', 0.01, 0.1)
    uniform_episode_training_cap = trial.suggest_int('uniform_episode_training_cap', 1500, 2000)
    
    logging.info(f"   Params: γ={gamma:.3f}, α={alpha:.2e}, batch={batch_size}, cap={uniform_episode_training_cap}")
    
    try:
        policy_net, target_net, returns, curriculum = deep_q_learning_exploration_diversity(
            epsilon=1.0,
            gamma=gamma,
            alpha=alpha,
            q_network=None,
            n_episodes=2500,
            uniform_episode_training_cap=uniform_episode_training_cap,
            pole_lengths=np.linspace(0.4, 1.8, 15),
            env_name='CartPole-v1',
            batch_size=batch_size,
            buffer_capacity=buffer_capacity,
            update_target_every=update_target_every,
            epsilon_decay=epsilon_decay,
            epsilon_min=epsilon_min
        )
        
        score = np.mean(returns[-500:])
        elapsed_min = (datetime.now() - trial_start).total_seconds() / 60
        
        logging.info(f"EXPLORATION Trial {trial_num}/10 DONE | Score: {score:.2f} | Time: {elapsed_min:.1f}min")
        return score
        
    except Exception as e:
        logging.error(f"EXPLORATION Trial {trial_num}/10 FAILED: {e}")
        return float('-inf')

def optimize_model(model_name, objective_func, n_trials, results_queue):
    """Run optimization and put results in queue"""
    start_time = datetime.now()
    logging.info(f"\n{'='*60}")
    logging.info(f" {model_name.upper()} - OPTIMIZATION STARTED")
    logging.info(f"{'='*60}\n")
    
    try:
        study = optuna.create_study(
            direction='maximize',
            pruner=optuna.pruners.MedianPruner(
                n_startup_trials=3,
                n_warmup_steps=500,
                interval_steps=100
            ),
            study_name=model_name
        )
        
        study.optimize(objective_func, n_trials=n_trials, n_jobs=1, show_progress_bar=False)
        
        elapsed_hours = (datetime.now() - start_time).total_seconds() / 3600
        
        logging.info(f"\n{'='*60}")
        logging.info(f"{model_name.upper()} - OPTIMIZATION COMPLETE")
        logging.info(f"   Best Score: {study.best_value:.2f}")
        logging.info(f"   Total Time: {elapsed_hours:.2f} hours")
        logging.info(f"   Best Params: {study.best_params}")
        logging.info(f"{'='*60}\n")
        
        results = {
            'model': model_name,
            'best_value': study.best_value,
            'best_params': study.best_params,
            'total_time_hours': elapsed_hours,
            'timestamp': datetime.now().isoformat()
        }
        
        with open(f'{model_name}_best_params.json', 'w') as f:
            json.dump(results, f, indent=2)
        
        results_queue.put(results)
        
    except Exception as e:
        logging.error(f"{model_name.upper()} FAILED: {e}")
        results_queue.put({'model': model_name, 'error': str(e)})

def parallel_hyperparameter_tuning(n_trials_per_model=10):
    logging.info("="*80)
    logging.info("STARTING THREADED HYPERPARAMETER OPTIMIZATION")
    logging.info(f"Trials per model: {n_trials_per_model}")
    logging.info(f"Episodes per trial: 2500")
    logging.info("="*80)
    
    models = [
        ('baseline', objective_baseline),
        ('strati_buff', objective_strati_buff),
        ('acl', objective_acl),
        ('exploration_diversity', objective_exploration_diversity)
    ]
    
    # Create queue for results
    results_queue = Queue()
    
    # Create and start threads
    threads = []
    for model_name, objective_func in models:
        thread = threading.Thread(
            target=optimize_model,
            args=(model_name, objective_func, n_trials_per_model, results_queue),
            name=f"{model_name}_thread"
        )
        thread.start()
        threads.append(thread)
        logging.info(f"Started thread for {model_name}")
    
    # Wait for all threads to complete
    for thread in threads:
        thread.join()
    
    # Collect results
    results = []
    while not results_queue.empty():
        results.append(results_queue.get())
    
    logging.info("\n" + "="*80)
    logging.info("ALL OPTIMIZATIONS COMPLETED")
    logging.info("="*80)
    
    logging.info("\nSUMMARY:")
    for result in results:
        if 'error' in result:
            logging.info(f"\n{result['model'].upper()}: FAILED - {result['error']}")
        else:
            logging.info(f"\n{result['model'].upper()}:")
            logging.info(f"  Best Score: {result['best_value']:.2f}")
            logging.info(f"  Time: {result['total_time_hours']:.2f}h")
            logging.info(f"  Best Params:")
            for key, value in result['best_params'].items():
                logging.info(f"    {key}: {value}")
    
    return results

def train_with_best_params(model_name):
    """Train a model with its best hyperparameters"""
    # Load best params
    with open(f'{model_name}_best_params.json', 'r') as f:
        data = json.load(f)
    
    params = data['best_params']
    logging.info(f"\nTraining {model_name} with best parameters...")
    
    if model_name == 'baseline':
        policy_net, target_net, returns = deep_q_learning(
            epsilon=1.0,
            gamma=params['gamma'],
            alpha=params['alpha'],
            q_network=None,
            n_episodes=2500,
            pole_lengths=np.linspace(0.4, 1.8, 15),
            env_name='CartPole-v1',
            batch_size=params['batch_size'],
            buffer_capacity=params['buffer_capacity'],
            update_target_every=params['update_target_every'],
            epsilon_decay=params['epsilon_decay'],
            epsilon_min=params['epsilon_min']
        )
    elif model_name == 'strati_buff':
        policy_net, target_net, returns = deep_q_learning_strati_buff(
            epsilon=1.0,
            gamma=params['gamma'],
            alpha=params['alpha'],
            q_network=None,
            n_episodes=2500,
            pole_lengths=np.linspace(0.4, 1.8, 15),
            env_name='CartPole-v1',
            batch_size=params['batch_size'],
            buffer_capacity=params['buffer_capacity'],
            update_target_every=params['update_target_every'],
            epsilon_decay=params['epsilon_decay'],
            epsilon_min=params['epsilon_min']
        )
    elif model_name == 'acl':
        policy_net, target_net, returns, _ = deep_q_learning_acl(
            epsilon=1.0,
            gamma=params['gamma'],
            alpha=params['alpha'],
            q_network=None,
            n_episodes=2500,
            uniform_episode_training_cap=params['uniform_episode_training_cap'],
            pole_lengths=np.linspace(0.4, 1.8, 15),
            env_name='CartPole-v1',
            batch_size=params['batch_size'],
            buffer_capacity=params['buffer_capacity'],
            update_target_every=params['update_target_every'],
            epsilon_decay=params['epsilon_decay'],
            epsilon_min=params['epsilon_min']
        )
    elif model_name == 'exploration_diversity':
        policy_net, target_net, returns, _ = deep_q_learning_exploration_diversity(
            epsilon=1.0,
            gamma=params['gamma'],
            alpha=params['alpha'],
            q_network=None,
            n_episodes=2500,
            uniform_episode_training_cap=params['uniform_episode_training_cap'],
            pole_lengths=np.linspace(0.4, 1.8, 15),
            env_name='CartPole-v1',
            batch_size=params['batch_size'],
            buffer_capacity=params['buffer_capacity'],
            update_target_every=params['update_target_every'],
            epsilon_decay=params['epsilon_decay'],
            epsilon_min=params['epsilon_min']
        )
    
    torch.save(policy_net.state_dict(), f"{model_name}_optimized_policy.pth")
    logging.info(f"Saved optimized model to {model_name}_optimized_policy.pth")
    
    return policy_net, returns


THREAD CONFIGURATION
PyTorch threads: 1
OMP_NUM_THREADS: 1



c:\Users\Teodora\OneDrive\Documents\GitHub\RL_Ass3\intro_rl_assignment_3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
# Setup logging
log_file = setup_logging()
logging.info(f"Logging to: {log_file}\n")
    
# Test threading first
print("Testing threading...")
def test_func(n):
        import time
        time.sleep(1)
        return n * 2
    
test_threads = []
for i in range(4):
    t = threading.Thread(target=lambda x: print(f"Thread {x} works!"), args=(i,))
    t.start()
    test_threads.append(t)
    
for t in test_threads:
    t.join()
    
print("Threading works! Starting optimization...\n")
    
    # Run optimization
results = parallel_hyperparameter_tuning(n_trials_per_model=10)
    
logging.info("\n ALL DONE!")

# STEP 2: Train final models with best hyperparameters (parallel)
logging.info("\n" + "=" * 80)
logging.info("TRAINING FINAL MODELS WITH BEST HYPERPARAMETERS")
logging.info("=" * 80)
    
train_with_best_params('baseline')
train_with_best_params('strati_buff')
train_with_best_params('acl')
train_with_best_params('exploration_diversity')
    
logging.info("\n" + "=" * 80)
logging.info("COMPLETE! All models optimized and trained.")

12:39:42 [MainThread     ] Logging to: training_20251016_123942.log

12:39:42 [MainThread     ] ================================================================================
12:39:42 [MainThread     ] STARTING THREADED HYPERPARAMETER OPTIMIZATION
12:39:42 [MainThread     ] Trials per model: 10
12:39:42 [MainThread     ] Episodes per trial: 2500
12:39:42 [MainThread     ] ================================================================================
12:39:42 [baseline_thread] 
12:39:42 [MainThread     ] Started thread for baseline
12:39:42 [baseline_thread]  BASELINE - OPTIMIZATION STARTED
12:39:42 [strati_buff_thread] 
12:39:42 [MainThread     ] Started thread for strati_buff
12:39:42 [baseline_thread] ============================================================

[I 2025-10-16 12:39:42,058] A new study created in memory with name: baseline
12:39:42 [strati_buff_thread]  STRATI_BUFF - OPTIMIZATION STARTED
12:39:42 [acl_thread     ] 
12:39:42 [MainThread     ] Started thread for acl

Testing threading...
Thread 0 works!
Thread 1 works!
Thread 2 works!
Thread 3 works!
Threading works! Starting optimization...



C:\Users\Teodora\AppData\Local\Temp\ipykernel_17960\3387781163.py:274: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  states_t = torch.FloatTensor(states)


Episode 0/2500 | Avg Reward: 8.0 | Epsilon: 0.999
Episode 0/2500 | Avg Reward: 25.0 | Epsilon: 0.995
Episode 0/2500 | Avg Reward: 26.0 | Epsilon: 0.999 | Probability Distribution{np.float64(0.4): 0.06666666666666667, np.float64(0.5): 0.06666666666666667, np.float64(0.6): 0.06666666666666667, np.float64(0.7): 0.06666666666666667, np.float64(0.8): 0.06666666666666667, np.float64(0.8999999999999999): 0.06666666666666667, np.float64(1.0): 0.06666666666666667, np.float64(1.1): 0.06666666666666667, np.float64(1.2): 0.06666666666666667, np.float64(1.2999999999999998): 0.06666666666666667, np.float64(1.4): 0.06666666666666667, np.float64(1.5): 0.06666666666666667, np.float64(1.6): 0.06666666666666667, np.float64(1.6999999999999997): 0.06666666666666667, np.float64(1.8): 0.06666666666666667} | 
Pole Length Performances{} | 

Episode 0/2500 | Avg Reward: 49.0 | Epsilon: 0.997
  Exploration Stats: Unvisited=14, Min visits=0, Max visits=1
  Prioritizing (least visited): ['0.40(0.067)', '0.50(0.067

14:44:18 [baseline_thread] BASELINE Trial 1/10 DONE | Score: 82.27 | Time: 124.6min
[I 2025-10-16 14:44:18,096] Trial 0 finished with value: 82.268 and parameters: {'gamma': 0.9537368257349809, 'alpha': 0.009860908625021817, 'batch_size': 32, 'buffer_capacity': 5000, 'update_target_every': 11, 'epsilon_decay': 0.9987831133994778, 'epsilon_min': 0.0339952624103068}. Best is trial 0 with value: 82.268.
14:44:18 [baseline_thread] BASELINE Trial 2/10 STARTED
--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\Teodora\AppData\Local\Programs\Python\Python313\Lib\logging\__init__.py", line 1154, in emit
    stream.write(msg + self.terminator)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Teodora\AppData\Local\Programs\Python\Python313\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encod

...

09:52:12 [MainThread     ] Saved optimized model to strati_buff_optimized_policy.pth
09:52:12 [MainThread     ] 
Training acl with best parameters...


Episode 0/2500 | Avg Reward: 26.0 | Epsilon: 0.997 | Probability Distribution{np.float64(0.4): 0.06666666666666667, np.float64(0.5): 0.06666666666666667, np.float64(0.6): 0.06666666666666667, np.float64(0.7): 0.06666666666666667, np.float64(0.8): 0.06666666666666667, np.float64(0.8999999999999999): 0.06666666666666667, np.float64(1.0): 0.06666666666666667, np.float64(1.1): 0.06666666666666667, np.float64(1.2): 0.06666666666666667, np.float64(1.2999999999999998): 0.06666666666666667, np.float64(1.4): 0.06666666666666667, np.float64(1.5): 0.06666666666666667, np.float64(1.6): 0.06666666666666667, np.float64(1.6999999999999997): 0.06666666666666667, np.float64(1.8): 0.06666666666666667} | 
Pole Length Performances{} | 

Episode 50/2500 | Avg Reward: 36.9 | Epsilon: 0.860 | Probability Distribution{np.float64(0.4): 0.06666666666666667, np.float64(0.5): 0.06666666666666667, np.float64(0.6): 0.06666666666666667, np.float64(0.7): 0.06666666666666667, np.float64(0.8): 0.06666666666666667, np.f

11:11:56 [MainThread     ] Saved optimized model to acl_optimized_policy.pth
11:11:56 [MainThread     ] 
Training exploration_diversity with best parameters...


Episode 0/2500 | Avg Reward: 31.0 | Epsilon: 0.996
  Exploration Stats: Unvisited=14, Min visits=0, Max visits=1
  Prioritizing (least visited): ['0.40(0.067)', '0.50(0.067)', '0.60(0.067)']
Episode 50/2500 | Avg Reward: 36.6 | Epsilon: 0.808
  Exploration Stats: Unvisited=0, Min visits=1, Max visits=6
  Prioritizing (least visited): ['0.40(0.067)', '0.50(0.067)', '0.60(0.067)']
Episode 100/2500 | Avg Reward: 40.8 | Epsilon: 0.655
  Exploration Stats: Unvisited=0, Min visits=3, Max visits=12
  Prioritizing (least visited): ['0.40(0.067)', '0.50(0.067)', '0.60(0.067)']
Episode 150/2500 | Avg Reward: 66.0 | Epsilon: 0.531
  Exploration Stats: Unvisited=0, Min visits=4, Max visits=15
  Prioritizing (least visited): ['0.40(0.067)', '0.50(0.067)', '0.60(0.067)']
Episode 200/2500 | Avg Reward: 112.4 | Epsilon: 0.431
  Exploration Stats: Unvisited=0, Min visits=9, Max visits=21
  Prioritizing (least visited): ['0.40(0.067)', '0.50(0.067)', '0.60(0.067)']
Episode 250/2500 | Avg Reward: 179.4 |

11:45:32 [MainThread     ] Saved optimized model to exploration_diversity_optimized_policy.pth
11:45:32 [MainThread     ] 
11:45:32 [MainThread     ] COMPLETE! All models optimized and trained.
